# Baseline Models Training
### CNN-Only | GRU-Only | CNN+GRU
Trains and saves checkpoints for all three baseline models.
Results are compared against TCN in `07_testing_comparison.ipynb`.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import warnings; warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt
import json, time
from pathlib import Path

from src.models import CNNClassifier, GRUClassifier, CNNGRUClassifier, FocalLoss
from src.training.trainer import run_epoch
from src.data.shared_loaders import get_loaders

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
train_loader, val_loader, test_loader, meta = get_loaders('config.yaml', subsampled=True)
print('DataLoaders ready.')
print(f'  Train : {meta["train"]["n"]:,}  seizure: {meta["train"]["n_seizure"]:,} ({meta["train"]["seizure_frac"]:.2%})')
print(f'  Val   : {meta["val"]["n"]:,}  seizure: {meta["val"]["n_seizure"]:,} ({meta["val"]["seizure_frac"]:.2%})')
print(f'  Test  : {meta["test"]["n"]:,}  seizure: {meta["test"]["n_seizure"]:,} ({meta["test"]["seizure_frac"]:.2%})')

---
## Model Definitions

In [ ]:
BASELINES = {
    'CNN-Only' : CNNClassifier  (n_channels=23, n_samples=1024, dropout=0.5),
    'GRU-Only' : GRUClassifier  (n_channels=23, hidden_size=128, num_layers=2, dropout=0.5),
    'CNN+GRU'  : CNNGRUClassifier(n_channels=23, dropout=0.5),
}

print('Baseline Architectures\n' + '='*45)
for name, m in BASELINES.items():
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {name:<12} : {params:>9,} parameters')

---
## Training Loop

In [ ]:
def train_model(name, model, train_loader, val_loader, device,
                lr=3e-4, weight_decay=1e-4, max_epochs=30, patience=8):
    ckpt_path = CKPT_DIR / f'{name.replace("+","_").replace("-","_")}_best.pt'
    log_path  = CKPT_DIR / f'{name.replace("+","_").replace("-","_")}_log.json'

    model = model.to(device)

    if ckpt_path.exists() and log_path.exists():
        print(f'  [{name}] Loading saved checkpoint.')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        return json.load(open(log_path))

    criterion = FocalLoss(alpha=0.85, gamma=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    history = {'train_loss':[], 'train_f1':[], 'val_loss':[], 'val_f1':[], 'best_epoch':0}
    best_val_f1, no_improve = 0.0, 0

    print(f'  [{name}] Training up to {max_epochs} epochs...')
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        tr = run_epoch(model, train_loader, criterion, optimizer, device, threshold=0.5)
        vl = run_epoch(model, val_loader,   criterion, None,      device, threshold=0.5)
        scheduler.step()

        history['train_loss'].append(tr['loss']); history['train_f1'].append(tr['f1'])
        history['val_loss'].append(vl['loss']);   history['val_f1'].append(vl['f1'])

        print(f'    Ep {epoch:02d}/{max_epochs}  '
              f'train loss={tr["loss"]:.4f} f1={tr["f1"]:.4f}  '
              f'val loss={vl["loss"]:.4f} f1={vl["f1"]:.4f}  ({time.time()-t0:.0f}s)')

        if vl['f1'] > best_val_f1:
            best_val_f1 = vl['f1']; history['best_epoch'] = epoch
            torch.save(model.state_dict(), ckpt_path); no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'    Early stop (best epoch {history["best_epoch"]})')
                break

    json.dump(history, open(log_path, 'w'), indent=2)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'    Best Val F1={best_val_f1:.4f} → {ckpt_path.name}')
    return history

histories = {}

---
## Train CNN-Only

In [ ]:
print('CNN-Only\n' + '='*45)
histories['CNN-Only'] = train_model('CNN-Only', BASELINES['CNN-Only'], train_loader, val_loader, DEVICE)

---
## Train GRU-Only

In [ ]:
print('GRU-Only\n' + '='*45)
histories['GRU-Only'] = train_model('GRU-Only', BASELINES['GRU-Only'], train_loader, val_loader, DEVICE)

---
## Train CNN+GRU

In [ ]:
print('CNN+GRU\n' + '='*45)
histories['CNN+GRU'] = train_model('CNN+GRU', BASELINES['CNN+GRU'], train_loader, val_loader, DEVICE)

---
## Training Curves

In [ ]:
colors = {'CNN-Only':'#e67e22', 'GRU-Only':'#2980b9', 'CNN+GRU':'#27ae60'}
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for col, (name, hist) in enumerate(histories.items()):
    epochs = range(1, len(hist['train_loss'])+1)
    c = colors[name]
    ax = axes[0, col]
    ax.plot(epochs, hist['train_loss'], '--', color=c, alpha=0.7, label='Train')
    ax.plot(epochs, hist['val_loss'],   '-',  color=c,            label='Val')
    ax.axvline(hist['best_epoch'], color='red', linestyle=':', label=f'Best ep {hist["best_epoch"]}')
    ax.set_title(f'{name} — Loss', fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1, col]
    ax.plot(epochs, hist['train_f1'], '--', color=c, alpha=0.7, label='Train')
    ax.plot(epochs, hist['val_f1'],   '-',  color=c,            label='Val')
    ax.axvline(hist['best_epoch'], color='red', linestyle=':')
    ax.set_title(f'{name} — F1 (best={max(hist["val_f1"]):.3f})', fontweight='bold')
    ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Baseline Models — Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Report_Figures/baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → Report_Figures/baseline_training_curves.png')

In [ ]:
print('Checkpoint Summary')
print('='*50)
for f in sorted(CKPT_DIR.iterdir()):
    print(f'  {f.name:<45}  {f.stat().st_size/1024:>7.1f} KB')

print('\nBaseline Training Summary:')
for name, hist in histories.items():
    print(f'  {name:<12}  epochs={len(hist["train_loss"]):>3}  '
          f'best_epoch={hist["best_epoch"]:>3}  best_val_f1={max(hist["val_f1"]):.4f}')